In [48]:
import numpy as np
import pyvista as pv


def plot_data(
        data: np.ndarray | list,
        size: float = 5.0,
):
    coords = data
    colors = np.clip(data, 0, 1)

    cloud = pv.PolyData(coords)
    cloud["colors"] = (colors * 255).astype(np.uint8)

    plotter = pv.Plotter()
    plotter.add_points(
        cloud,
        scalars="colors",
        rgb=True,
        point_size=size
    )

    plotter.add_axes()
    plotter.show_grid()
    plotter.show()


def plot_clustered_pv(
        data: np.ndarray,
        labels: np.ndarray,
        cluster_centers: np.ndarray,
        size: float = 4.0
):
    coords = data
    cluster_colors = cluster_centers[labels]
    colors = np.clip(cluster_colors, 0, 1)

    cloud = pv.PolyData(coords)
    cloud["colors"] = (colors * 255).astype(np.uint8)

    plotter = pv.Plotter()
    plotter.add_points(
        cloud,
        scalars="colors",
        rgb=True,
        point_size=size
    )

    plotter.add_axes()
    plotter.show_grid()
    plotter.show()


def plot_centroids_pv(
        cluster_centers: np.ndarray,
        size: float = 18.0
):
    coords = cluster_centers
    colors = np.clip(cluster_centers, 0, 1)

    cloud = pv.PolyData(coords)
    cloud["colors"] = (colors * 255).astype(np.uint8)

    plotter = pv.Plotter()
    plotter.add_points(
        cloud,
        scalars="colors",
        rgb=True,
        point_size=size,
        render_points_as_spheres=True
    )

    plotter.add_axes()
    plotter.show_grid()
    plotter.show()

In [49]:
import numpy as np

def generate_dataset(centers, stds, n_samples=500, ndim=3):
    X = []
    y = []

    for i, center in enumerate(centers):
        cluster = np.random.normal(loc=center, scale=stds[i], size=(n_samples // len(centers), ndim))
        X.append(cluster)
        y.append(np.full(n_samples // len(centers), i))

    return np.vstack(X), np.concatenate(y)

In [50]:
import pandas as pd
import pyvista as pv
import matplotlib.pyplot as plt

default_colors = np.array(plt.colormaps.get_cmap('tab10').colors)

In [51]:
data, ground_truth = generate_dataset(
    [[1, 1, 1], [3, 3, 3], [2, 2, 1]],
    stds=[.5, .5, .5],
    n_samples=500
)

pd.DataFrame(data)

,0,1,2
0,1.394050,1.235366,0.622998
1,0.609271,0.196924,0.723076
2,1.825054,1.240620,1.087707
3,0.696686,1.290818,0.720545
4,0.387395,0.745999,0.648529
...,...,...,...
493,1.062370,1.822287,1.758156
494,1.506145,1.677183,2.017359
495,1.615590,0.923905,1.043505
496,2.457671,1.951824,0.468887


In [52]:
plt = pv.Plotter()

for n, d in zip(ground_truth, data):
    cloud = pv.PolyData(d)
    color = default_colors[n % len(default_colors)]

    plt.add_points(
        cloud,
        color=color,
        render_points_as_spheres=True,
        smooth_shading=True,
        point_size=10
    )

plt.add_axes()
plt.show_grid()
plt.show()

Widget(value='<iframe src="http://localhost:42417/index.html?ui=P_0x7f2e50f2b9d0_23&reconnect=auto" class="pyv…

In [53]:
from ktree.ntree import NTreeDynamic

tree = NTreeDynamic(2)

for a in data:
    tree.insert(a)

sorted_data = tree.sort()

In [54]:
plt = pv.Plotter()

for n, cluster in enumerate(sorted_data):
    s_data = list(cluster)
    cloud = pv.PolyData(s_data)
    color = default_colors[n % len(default_colors)]

    centroid = np.mean(s_data, axis=0)

    plt.add_points(
        cloud,
        color=color,
        render_points_as_spheres=True,
        smooth_shading=True,
        point_size=5
    )

    plt.add_points(
        centroid,
        color=color,
        render_points_as_spheres=True,
        smooth_shading=True,
        point_size=len(s_data) // 2
    )


plt.add_axes()
plt.show_grid()
plt.show()

Widget(value='<iframe src="http://localhost:42417/index.html?ui=P_0x7f2e50f2a5d0_24&reconnect=auto" class="pyv…

In [59]:
n_clusters = 3

all_clusters = sorted(sorted_data, key=lambda x: len(x))[::-1]
clusters = all_clusters[:n_clusters]
all_data = all_clusters[n_clusters:]

main_clusters = []

for cluster in clusters:
    c_centroid = np.mean([*cluster], axis=0)
    sum_dist = 0

    sub_cluster = []

    for data in all_data:
        d_centroid = np.mean([*data], axis=0)
        dist = np.linalg.norm(c_centroid - d_centroid)

        sum_dist += dist

        sub_cluster.append((dist, data))

    mead_dist = sum_dist / len(all_data)
    sub_cluster = sorted(sub_cluster, key=lambda a: a[0])

    main_clusters.append((cluster, [data for (dist, data) in sub_cluster if dist < mead_dist]))


for (cluster, all_data) in main_clusters:
    print("Cluster: ", cluster)
    print("Data: ", all_data)


Cluster:  Cluster(axis=[[0.9857819659288977, 2.105376260792815], [0.963347972829719, 2.099233778265928], [0.7939324127899722, 1.7985901279808314]])
Data:  [Cluster(axis=[[1.051871418689509, 1.3954467593063633], [2.1455398601930176, 2.2464488591989884], [0.8478927734520884, 1.1444870092794737]]), Cluster(axis=[[2.141986232823574, 2.579591741314684], [1.4649830253341602, 2.097929394391985], [0.9932023802519806, 1.7288393404201752]]), Cluster(axis=[[1.495573882148397, 2.098485287619024], [2.141932410990679, 2.6600748331622643], [1.0159128142111942, 1.8352873844268993]]), Cluster(axis=[[1.1781065715567756, 1.5061454575745588], [1.287121814109204, 1.6771826351424903], [2.0173593922427635, 2.161033690206296]]), Cluster(axis=[[1.0558573441241872, 1.0558573441241872], [1.1375510048244983, 1.1375510048244983], [1.855728091051776, 1.855728091051776]]), Cluster(axis=[[1.41615796560929, 2.041341694376182], [2.1199800970196985, 2.6377487084850104], [-0.3147130736627515, 0.7224225635489867]]), Clust

In [60]:
plt = pv.Plotter()

for n, (cluster, all_data) in enumerate(main_clusters):
    s_data = list(cluster)

    cloud = pv.PolyData(list(cluster))
    color = default_colors[n % len(default_colors)]

    centroid = np.mean(s_data, axis=0)

    plt.add_points(
        centroid,
        color=color,
        render_points_as_spheres=True,
        smooth_shading=True,
        point_size=25
    )

    for data in (all_data + [cluster]):
        cloud = pv.PolyData(list(data))

        plt.add_points(
            cloud,
            color=color,
            render_points_as_spheres=True,
            smooth_shading=True,
            point_size=10
        )


plt.add_axes()
plt.show_grid()
plt.show()

Widget(value='<iframe src="http://localhost:42417/index.html?ui=P_0x7f2e44c116d0_27&reconnect=auto" class="pyv…